# **SESSION 2.2 - PRACTICE PROJECT**

## **🎯 GOAL: Train a Speaker Count Classifier**

Build and train a neural network that predicts how many speakers are in an audio clip (1, 2, or 3 speakers).

---

## **📋 PROJECT REQUIREMENTS**

### **Input:**
- Audio mixture tensor from your dataset
- Extract 5 features: mean, std, min, max, energy

### **Output:**
- Class prediction: 0 (1 speaker), 1 (2 speakers), or 2 (3 speakers)

### **Model:**
- 2-3 layer network (your choice of architecture)
- Use ReLU activations in hidden layers
- Output layer has 3 neurons (one per class)

### **Training:**
- Train for 10-20 epochs
- Use your training data (load multiple samples)
- Compute loss and update weights
- Track training progress

### **Evaluation:**
- Test on validation samples
- Report accuracy
- Show confusion: which classes does it confuse?

---

## **📐 ARCHITECTURE SUGGESTIONS**

You decide the architecture! Here are some options:

**Option A: Small Network (Fast)**
```
5 features → 16 neurons → 8 neurons → 3 classes
```

**Option B: Medium Network (Balanced)**
```
5 features → 32 neurons → 16 neurons → 3 classes
```

**Option C: Deeper Network (More capacity)**
```
5 features → 64 neurons → 32 neurons → 16 neurons → 3 classes
```

**You choose!** Start simple, you can always make it bigger.

---

## **🛠️ WHAT I GIVE YOU**

### **1. Optimizer (Weight Update Rule)**

```python
class SimpleOptimizer:
    """Simple SGD optimizer - updates weights using gradients"""
    def __init__(self, parameters, lr=0.01):
        self.parameters = parameters  # List of [W1, b1, W2, b2, ...]
        self.lr = lr
    
    def step(self):
        """Update all parameters using their gradients"""
        for param in self.parameters:
            if param.grad is not None:
                param.data = param.data - self.lr * param.grad
    
    def zero_grad(self):
        """Clear all gradients"""
        for param in self.parameters:
            if param.grad is not None:
                param.grad.zero_()
```

**Usage:**
```python
# After loss.backward()
optimizer.step()  # Update weights
optimizer.zero_grad()  # Clear gradients
```

---

### **2. Helper: Compute Accuracy**

```python
def compute_accuracy(predictions, labels):
    """
    predictions: (batch_size, 3) logits
    labels: (batch_size,) true classes
    
    Returns: accuracy as percentage
    """
    predicted_classes = torch.argmax(predictions, dim=1)
    correct = (predicted_classes == labels).sum().item()
    total = labels.size(0)
    return 100.0 * correct / total
```

---

### **3. Data Loading Template**

```python
def load_training_batch(manifest_path, batch_size=10):
    """
    Load a batch of audio samples for training.
    
    Returns:
        features: (batch_size, 5) - extracted features
        labels: (batch_size,) - speaker counts (0, 1, or 2)
    """
    # YOUR CODE: Load samples, extract features, get labels
    pass
```

---

## **🎯 YOUR TASKS**

### **Task 1: Build the Model**

Create your network architecture with proper initialization.

**Hints:**
- Initialize weights with `torch.randn()` (but scale them!)
- Set `requires_grad=True` on all parameters
- Store all parameters in a list for the optimizer

**Structure:**
```python
# Define architecture sizes
input_size = 5
hidden1_size = ???  # You choose
hidden2_size = ???  # You choose (optional)
output_size = 3

# Initialize weights
W1 = ???
b1 = ???
# ... more layers

# Collect parameters
parameters = [W1, b1, W2, b2, ...]  # All tensors with requires_grad=True
```

---

### **Task 2: Implement Data Loading**

Complete the `load_training_batch()` function.

**Hints:**
- Use your `load_audio_with_labels()` function
- Extract features from each mixture
- Get speaker count from metadata (subtract 1 for 0-indexing)
- Stack into batches

**Pseudocode:**
```
1. Load batch_size samples from manifest
2. For each sample:
   - Extract 5 features
   - Get num_speakers - 1 as label
3. Stack features into (batch_size, 5) tensor
4. Stack labels into (batch_size,) tensor
5. Return both
```

---

### **Task 3: Implement Forward Pass**

Create a function that runs the full network.

```python
def forward(x, parameters):
    """
    x: (batch_size, 5) features
    parameters: [W1, b1, W2, b2, ...]
    
    Returns: (batch_size, 3) logits
    """
    # YOUR CODE: Pass through all layers
    pass
```

**Hints:**
- Use your `two_layer_network()` logic but handle batches
- For batches: `W @ x.T` gives (output_size, batch_size), then transpose back
- Or loop through batch (simpler but slower)

---

### **Task 4: Training Loop**

Put it all together!

**Template:**
```python
# Setup
learning_rate = 0.01
num_epochs = 20
batch_size = 10

optimizer = SimpleOptimizer(parameters, lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    # Load batch
    features, labels = load_training_batch(manifest_path, batch_size)
    
    # Forward pass
    logits = forward(features, parameters)
    
    # Compute loss for each sample, then average
    total_loss = 0
    for i in range(batch_size):
        loss = cross_entropy_loss(logits[i], labels[i])
        total_loss += loss
    avg_loss = total_loss / batch_size
    
    # Backward pass
    avg_loss.backward()
    
    # Update weights
    optimizer.step()
    optimizer.zero_grad()
    
    # Compute accuracy
    accuracy = compute_accuracy(logits, labels)
    
    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss.item():.4f}, Accuracy: {accuracy:.2f}%")
```

---

### **Task 5: Evaluation**

Test on validation data.

```python
# Load validation batch
val_features, val_labels = load_training_batch(val_manifest_path, batch_size=20)

# Forward pass (no gradient needed)
with torch.no_grad():
    val_logits = forward(val_features, parameters)
    val_accuracy = compute_accuracy(val_logits, val_labels)

print(f"\nValidation Accuracy: {val_accuracy:.2f}%")

# Confusion analysis
predictions = torch.argmax(val_logits, dim=1)
for true_class in range(3):
    mask = val_labels == true_class
    if mask.sum() > 0:
        pred_for_class = predictions[mask]
        print(f"True class {true_class} ({true_class+1} speakers):")
        print(f"  Predicted as {pred_for_class.tolist()}")
```

---

## **📊 EXPECTED OUTPUT**

```
Epoch 5/20, Loss: 0.8234, Accuracy: 45.00%
Epoch 10/20, Loss: 0.5123, Accuracy: 70.00%
Epoch 15/20, Loss: 0.3456, Accuracy: 85.00%
Epoch 20/20, Loss: 0.2341, Accuracy: 95.00%

Validation Accuracy: 80.00%

True class 0 (1 speaker):
  Predicted as [0, 0, 0, 1]
True class 1 (2 speakers):
  Predicted as [1, 1, 1, 1, 1]
True class 2 (3 speakers):
  Predicted as [2, 2, 1, 2, 2]
```

---

## **💡 HINTS & TIPS**

### **Batch Processing:**
If handling batches is tricky, start with batch_size=1, then upgrade to batching later.

### **Weight Initialization:**
```python
# Good initialization (Xavier/He initialization)
W1 = torch.randn(hidden_size, input_size) * 0.1
W1.requires_grad = True
```

### **Debugging:**
```python
# Print shapes to debug
print(f"Features shape: {features.shape}")
print(f"W1 shape: {W1.shape}")
print(f"After W1: {(W1 @ features.T).shape}")
```

### **If Loss Doesn't Decrease:**
- Check learning rate (try 0.001 or 0.1)
- Check weight initialization (not too large)
- Print gradients: `print(W1.grad)`
- Verify labels are correct (0, 1, 2)

---

## **🎯 DELIVERABLES**

Show me:

1. ✅ **Your architecture choice** (how many layers, how many neurons)
2. ✅ **Training output** (loss and accuracy over epochs)
3. ✅ **Validation accuracy** (how well it generalizes)
4. ✅ **One insight** you discovered (what worked? what didn't?)

---

## **BONUS CHALLENGES (Optional)**

1. **Plot training curve** - Loss vs epoch
2. **Try different learning rates** - Which works best?
3. **Compare architectures** - Small vs large network
4. **Feature engineering** - Try different audio features

---

**You have everything you need!** 

This is YOUR project. I'll only help if you get stuck. Take your time, experiment, and learn from failures!

**Ready? Start coding!** 🚀

Come back when you have results or questions!

In [1105]:
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
from pathlib import Path
import os
from typing import List, Tuple, Dict
import sys
import json
import random

# Check if __file__ exists (it won't in Jupyter)
try:
    current_dir = Path(__file__).parent
except NameError:
    # If in Jupyter, use the current working directory
    current_dir = Path(os.getcwd())

# Add project root to Python path
project_root = current_dir.parent.parent
sys.path.insert(0, str(project_root))

from src.preprocessing.audio_utils import load_audio, get_audio_duration

# Device configuration (for your MacBook)
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print("✅ Using Apple Silicon GPU")
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print("✅ Using NVIDIA GPU")
else:
    device = torch.device('cpu')
    print("⚠️ Using CPU")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

✅ Using Apple Silicon GPU
PyTorch version: 2.10.0
Device: mps


## neural network setup

In [1106]:
input_size = 5
h1_size = 64
h2_size = 32
h3_size = 16
output_size = 3

# Replace lines 83-90
import math

def init_weights(input_size, output_size):
    """Xavier initialization"""
    limit = math.sqrt(6.0 / (input_size + output_size))
    W = torch.empty(output_size, input_size).uniform_(-limit, limit)
    b = torch.zeros(output_size)
    return W, b

# Initialize all layers
W1, b1 = init_weights(input_size, h1_size)
W2, b2 = init_weights(h1_size, h2_size)
W3, b3 = init_weights(h2_size, h3_size)
W4, b4 = init_weights(h3_size, output_size)

# Move to device
W1, b1 = W1.to(device), b1.to(device)
W2, b2 = W2.to(device), b2.to(device)
W3, b3 = W3.to(device), b3.to(device)
W4, b4 = W4.to(device), b4.to(device)

W1.requires_grad = True
W2.requires_grad = True
W3.requires_grad = True
W4.requires_grad = True
b1.requires_grad = True
b2.requires_grad = True
b3.requires_grad = True
b4.requires_grad = True

parameters = [W1, b1, W2, b2, W3, b3, W4, b4]

## data loading 

In [1107]:
def pad_or_trim_audio(audio_tensor: torch.Tensor, target_length: int) -> torch.Tensor:
    """Pad with zeros or trim audio to target length."""
    current_length = audio_tensor.shape[0]
    
    if current_length < target_length:
        padding = target_length - current_length
        audio_tensor = torch.nn.functional.pad(audio_tensor, (0, padding))
    elif current_length > target_length:
        audio_tensor = audio_tensor[:target_length]
    
    return audio_tensor

In [1108]:
def feature_extraction(
        audios: list[torch.Tensor]
) -> torch.Tensor:
        output = []
        for audio in audios:
                mean = torch.mean(audio)
                std = torch.std(audio)
                min = torch.min(audio)
                max = torch.max(audio)
                square = audio ** 2
                energy = torch.mean(square)
        
                output.append(torch.stack([mean, std, min, max, energy]))

        return torch.stack(output)

In [1109]:
def load_training_batch(
    manifest_path: Path,
    num_samples: int = 5,
    device: str = 'cpu'
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, List[Dict]]:
    """
    Load audio mixtures, clean sources, and diarization labels from manifest.
    
    Args:
        manifest_path: Path to train_manifest.json
        num_samples: How many samples to load
        target_length: Target length in samples (pad/trim to this)
        max_speakers: Maximum number of speakers (for padding speaker dimension)
        device: Device to move tensors to
    
    Returns:
        mixtures: (batch, 1, target_length) - Mixed audio
        sources: (batch, max_speakers, target_length) - Clean sources per speaker
        labels: (batch, max_speakers, num_frames) - Diarization labels
        metadata: List of dict with original info
    
    YOUR TASK:
    Load your generated dataset and prepare it for training!
    """
    
    # TODO 1: Load manifest JSON
    # Hint: json.load()
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)
    
    # Take only num_samples entries
    manifest = manifest[:num_samples]
    
    # Initialize lists
    mixture_tensors = []
    label_tensors = []
    
    # TODO 2: Loop through each entry in manifest
    for entry in manifest:
        # TODO 2.1: Load mixture audio
        mixture_path = Path(entry['mixture_path'])
        mixture_audio, sr = load_audio(mixture_path)  # Use load_audio
        
        # TODO 2.2: Convert to tensor and pad/trim
        mixture_tensor = torch.from_numpy(mixture_audio)
        mixture_tensor = mixture_tensor.view(1, 1, -1)  # (1, 1, samples)
        mixture_tensor = feature_extraction(mixture_tensor)
        mixture_tensors.append(mixture_tensor)
        
        
        # TODO 2.6: Convert labels to tensor
        # Shape is (num_frames, num_speakers)
        # We need (max_speakers, num_frames) with padding
        speaker_count = int(entry['num_speakers']) - 1
        label_tensor = torch.tensor(speaker_count, dtype=torch.long)
        
        label_tensors.append(label_tensor)
        
    
    # TODO 3: Stack all tensors into batches
    mixtures_batch = torch.cat(tensors=mixture_tensors, dim=0).to(device)  # torch.cat along dim=0
    labels_batch = torch.stack(label_tensors).to(device)    # torch.stack
    
    return mixtures_batch, labels_batch


In [1110]:
train_manifest_path = Path.cwd().parent.parent / "data" / "processed" / "train" / "train_manifest.json"
test_manifest_path = Path.cwd().parent.parent / "data" / "processed" / "test" / "test_manifest.json"

In [1111]:
def forward(
        features: torch.Tensor,
        parameters: list
    ) -> torch.Tensor:
    """
    Optimized forward pass using x @ W.T + b
    features shape: (batch_size, 5)
    """
    x = features
    
    # Iterate through all hidden layers
    for i in range(0, len(parameters) - 2, 2):
        W = parameters[i]     # Shape: (out_features, in_features)
        b = parameters[i+1]   # Shape: (out_features)
        
        # Linear transformation: (batch, in) @ (in, out) + (out)
        x = x @ W.T + b
        x = torch.relu(x)
        
    # Final Output Layer (No ReLU)
    w_final = parameters[-2]
    b_final = parameters[-1]
    
    logits = x @ w_final.T + b_final
    
    return logits # Final shape: (batch_size, 3)

## Training Loop

In [1112]:
class SimpleOptimizer:
    """Simple SGD optimizer - updates weights using gradients"""
    def __init__(self, parameters, lr=0.01):
        self.parameters = parameters  # List of [W1, b1, W2, b2, ...]
        self.lr = lr
    
    def step(self):
        """Update all parameters using their gradients"""
        for param in self.parameters:
            if param.grad is not None:
                param.data = param.data - self.lr * param.grad
    
    def zero_grad(self):
        """Clear all gradients"""
        for param in self.parameters:
            if param.grad is not None:
                param.grad.zero_()

In [1113]:
def compute_accuracy(predictions, labels):
    """
    predictions: (batch_size, 3) logits
    labels: (batch_size,) true classes
    
    Returns: accuracy as percentage
    """
    predicted_classes = torch.argmax(predictions, dim=1)
    correct = (predicted_classes == labels).sum().item()
    total = labels.size(0)
    return 100.0 * correct / total

In [1114]:
from torch.nn.functional import softmax

def cross_entropy_loss(logits, target):
    prob = softmax(logits, dim=0)
    correct_prob = prob[target]
    return -torch.log(correct_prob)

In [1115]:
# Setup
learning_rate = 0.1
num_epochs = 50
batch_size = 10

optimizer = SimpleOptimizer(parameters, lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    # Load batch
    features, labels = load_training_batch(train_manifest_path, batch_size, device=device)
    
    # Forward pass
    logits = forward(features, parameters)
    
    # Compute loss for each sample, then average
    loss = F.cross_entropy(logits, labels)
    
    # Backward pass
    loss.backward()
    
    # Update weights
    optimizer.step()
    optimizer.zero_grad()
    
    # Compute accuracy
    accuracy = compute_accuracy(logits, labels)
    
    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}, Accuracy: {accuracy:.2f}%")

Epoch 5/50, Loss: 0.9822, Accuracy: 60.00%
Epoch 10/50, Loss: 0.8968, Accuracy: 60.00%
Epoch 15/50, Loss: 0.8390, Accuracy: 60.00%
Epoch 20/50, Loss: 0.7986, Accuracy: 60.00%
Epoch 25/50, Loss: 0.7689, Accuracy: 60.00%
Epoch 30/50, Loss: 0.7468, Accuracy: 60.00%
Epoch 35/50, Loss: 0.7305, Accuracy: 60.00%
Epoch 40/50, Loss: 0.7182, Accuracy: 60.00%
Epoch 45/50, Loss: 0.7091, Accuracy: 60.00%
Epoch 50/50, Loss: 0.7022, Accuracy: 60.00%


In [1116]:
# Load validation batch
test_features, test_labels = load_training_batch(test_manifest_path, 20)
test_features = test_features.to(device)
test_labels = test_labels.to(device)

# Forward pass (no gradient needed)
with torch.no_grad():
    val_logits = forward(test_features, parameters)
    val_accuracy = compute_accuracy(val_logits, test_labels)

print(f"\nValidation Accuracy: {val_accuracy:.2f}%")

# Confusion analysis
predictions = torch.argmax(val_logits, dim=1)
for true_class in range(3):
    mask = test_labels == true_class
    if mask.sum() > 0:
        pred_for_class = predictions[mask]
        print(f"True class {true_class} ({true_class+1} speakers):")
        print(f"  Predicted as {pred_for_class.tolist()}")


Validation Accuracy: 55.00%
True class 1 (2 speakers):
  Predicted as [2, 2, 2, 2, 2, 2, 2, 2, 2]
True class 2 (3 speakers):
  Predicted as [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
